<a href="https://colab.research.google.com/github/mwoonkim/Learn-ML/blob/main/03_MNIST_Dataset_Load_with_tensorflow.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [25]:
import tensorflow as tf
from tensorflow.keras.datasets import mnist
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

In [2]:
import sys, os
import numpy as np
import pickle

In [ ]:
# MNIST 데이터셋 다운로드 및 로딩 - 처음에만 수행하고 이후에는 pickle을 사용하여 불러옴
(x_train, t_train), (x_test, t_test) = mnist.load_data()

In [ ]:
# 각 자료의 갯수와 array 형태를 출력하여 확인
print(f"학습 데이터 형태: {x_train.shape}")    # 60000개의 28 x 28 pixel data
print(f"학습 라벨 형태: {t_train.shape}")      # 60000개의 data
print(f"테스트 데이터 형태: {x_test.shape}")   # 0000개의 28 x 28 pixel data
print(f"테스트 라벨 형태: {t_test.shape}")     # 10000개의 data

In [29]:
# [option] normalize (0 ~ 255 범위의 pixel 값을 0 ~ 1.0 사이의 값으로 변환)
x_train = x_train.astype(np.float32) / 255.0
x_test = x_test.astype(np.float32) / 255.0

In [ ]:
# 불러온 이미지 확인 (Python Image Library 모듈 사용)
from PIL import Image
def img_show(img) :
  pil_img = Image.fromarray(np.uint8(img))  # 8 bit 정수(0 ~ 255의 값)로 변환
  pil_img.show()

img = x_train[0]
label = t_train[0]
print(label)    # 이미지의 라벨(숫자)을 확인

print(img.shape)
img = img.reshape(28,28)  # 이미지를 1차원의 numpy 배열로 저장했을 경우에 다시 원래 크기로 변형
print(img.shape)

img_show(img)
mpimg.imsave('image.jpg',img)  # 화면으로 보이지 않을 경우, 파일로 저장

In [31]:
# sample_weight.pkl 파일(학습된 가중치 매개변수값)을 읽어들임
with open("/content/drive/MyDrive/Colab Notebooks/sample_weight.pkl", 'rb') as f:
  network = pickle.load(f)

In [32]:
def sigmoid(x):  # sigmoid 함수 정의
    return 1 / (1 + np.exp(-x))

def softmax(x):  # softmax 함수 정의
    x = x - np.max(x, axis=-1, keepdims=True) # 오버플로 방지 대책
    return np.exp(x) / np.sum(np.exp(x), axis=-1, keepdims=True)

In [36]:
# 추론 시행 정의
def predict(network, x):
    W1, W2, W3 = network['W1'], network['W2'], network['W3']
    b1, b2, b3 = network['b1'], network['b2'], network['b3']

    a1 = np.dot(x, W1) + b1   # 각 레이블(숫자)에 해당할 확률을 numpy 배열로 변환
    z1 = sigmoid(a1)
    a2 = np.dot(z1, W2) + b2
    z2 = sigmoid(a2)
    a3 = np.dot(z2, W3) + b3
    y = softmax(a3)

    return y

In [ ]:
# 추론의 정확도 평가
xt, tt = x_test, t_test

accuracy_cnt = 0
for i in range(len(xt)):                       # 이미지 데이터를 하나씩 꺼내어 추론하여 분류
    yt = predict(network, xt[i].reshape(784))  # 28 x 28의 값을 1차원으로 변환하여 추론함수에 적용
    p= np.argmax(yt) # 확률이 가장 높은 원소의 인덱스를 얻는다.
    if p == tt[i]:
        accuracy_cnt += 1

print("Accuracy:" + str(float(accuracy_cnt) / len(xt)))